Crear la base de datos a partir de `dssoporte.csv`

Este notebook:
1. Carga el dataset público `dssoporte.csv`.
2. Normaliza nombres de columnas y tipos de datos.
3. Crea la base de datos SQLite `dssoporte.db` con un esquema SQL explícito.
4. Exporta el script `dssoporte.sql`.

No se usa ningún token ni credencial.

In [1]:
import os, re, shutil, sqlite3
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')   # autoriza el acceso con tu cuenta de Google

BASE_DIR = '/content/drive/MyDrive/proyecto_soporte'
DATA_DIR = f'{BASE_DIR}/data'
SQL_DIR  = f'{BASE_DIR}/sql'
DB_DIR   = f'{BASE_DIR}/db'
for d in (DATA_DIR, SQL_DIR, DB_DIR):
    os.makedirs(d, exist_ok=True)

CSV_PATH = f'{DATA_DIR}/dssoporte.csv'
if not os.path.exists(CSV_PATH):
    from google.colab import files
    print('No se encontró dssoporte.csv en Drive. Súbelo ahora:')
    files.upload()
    shutil.move('dssoporte.csv', CSV_PATH)

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

ValueError: mount failed

## 2. Revisión inicial

In [ ]:
df.info()
print()
print('Valores nulos por columna:')
print(df.isna().sum())

## 3. Normalizar nombres de columnas
Los nombres con espacios (`Ticket ID`, `Created time`...) pasan a formato SQL válido (`ticket_id`, `created_time`...).

In [ ]:
df.columns = [re.sub(r'[^0-9a-zA-Z]+', '_', c.strip()).lower().strip('_') for c in df.columns]
print(list(df.columns))

## 4. Ajustar tipos de datos

In [ ]:
cols_fecha = ['created_time', 'expected_sla_to_resolve', 'expected_sla_to_first_response',
              'first_response_time', 'resolution_time', 'close_time']
for c in cols_fecha:
    df[c] = pd.to_datetime(df[c], errors='coerce', dayfirst=True).dt.strftime('%Y-%m-%d %H:%M:%S')

for c in ['agent_interactions', 'survey_results', 'latitude', 'longitude']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['ticket_id'] = pd.to_numeric(df['ticket_id'], errors='coerce')
df = df.dropna(subset=['ticket_id'])
df['ticket_id'] = df['ticket_id'].astype('int64')

print('Duplicados por ticket_id:', df.duplicated('ticket_id').sum())
df = df.drop_duplicates('ticket_id')
print(df.shape)

## 5. Crear la base de datos SQLite con esquema SQL explícito

In [ ]:
# SQLite se construye en el disco local de Colab (más estable) y luego se copia a Drive
LOCAL_DB = '/content/dssoporte.db'
if os.path.exists(LOCAL_DB):
    os.remove(LOCAL_DB)
conn = sqlite3.connect(LOCAL_DB)
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS tickets;
CREATE TABLE tickets (
    ticket_id INTEGER PRIMARY KEY,
    status TEXT,
    priority TEXT,
    source TEXT,
    topic TEXT,
    agent_group TEXT,
    agent_name TEXT,
    created_time TEXT,
    expected_sla_to_resolve TEXT,
    expected_sla_to_first_response TEXT,
    first_response_time TEXT,
    sla_for_first_response TEXT,
    resolution_time TEXT,
    sla_for_resolution TEXT,
    close_time TEXT,
    agent_interactions INTEGER,
    survey_results REAL,
    product_group TEXT,
    support_level TEXT,
    country TEXT,
    latitude REAL,
    longitude REAL
);
CREATE INDEX idx_tickets_status ON tickets(status);
CREATE INDEX idx_tickets_country ON tickets(country);
""")

df.to_sql('tickets', conn, if_exists='append', index=False)
conn.commit()
print('Base de datos creada.')

## 6. Verificar con consultas SQL

In [ ]:
print(pd.read_sql('SELECT COUNT(*) AS total_tickets FROM tickets', conn))
pd.read_sql('SELECT status, COUNT(*) AS n FROM tickets GROUP BY status ORDER BY n DESC', conn)

In [ ]:
pd.read_sql('SELECT * FROM tickets LIMIT 5', conn)

## 7. Exportar el script SQL y guardar la base en Drive

In [ ]:
SQL_PATH = f'{SQL_DIR}/dssoporte.sql'
DB_PATH  = f'{DB_DIR}/dssoporte.db'

with open(SQL_PATH, 'w', encoding='utf-8') as f:
    for linea in conn.iterdump():
        f.write(linea + '\n')

conn.close()
shutil.copy(LOCAL_DB, DB_PATH)
print('Guardado en Drive:')
print(' -', SQL_PATH)
print(' -', DB_PATH)

Siguiente paso: continúa con `02_analisis_y_visualizaciones.ipynb`. Para subir el notebook a GitHub descárgalo con *Archivo > Descargar > .ipynb* (o desde Drive).